# Exploring the Basener-Sanford model

Supporting material for "Basener and Sanford disprove population genetics – or do they?" by Joe Felsenstein and <a href="mailto:Tom.English.PhD@gmail.com">Tom English</a>.

In [1]:
!TZ=America/Los_Angeles date

Wed Jan 16 11:04:06 PST 2019


Our description of the model is customized to Sections 5.3 and 5.4 of W. F. Basener and J. C. Sanford, "<a href="https://doi.org/10.1007/s00285-017-1190-x">The fundamental theorem of natural selection with mutations</a>," J. Math. Biol. (2018) 76:1589–1622. For full generality, see Section 3 of their article.

**General setup**

In [2]:
%matplotlib notebook
"""
Load the code base.
"""
%run ../Code/bs.py
%run -i ../Code/multiprecision_gamma.py
%run -i ../Code/gamma.py
"""
Suppress automatic display of graphics generated by Matplotlib. All
graphics are saved to disk, and are displayed by explicit commands.
"""
plt.interactive(False)
"""
Define the name of the subdirectory that holds various files associated
with this notebook, and create the directory if it does not exist already.
"""
DIR = 'Introduction/'
!if [ ! -d '{DIR}' ]; then mkdir '{DIR}'; fi

## Sketch of discrete-fitness model without mutation

**Class.** Organisms are classified according to fitness of their genomes. The $i$-th class has frequency $P(i, t)$ at time $t.$ We usually write just $P_i.$ Frequencies change continuously in time.

**Population.** The population is represented as vector $[P_0, P_1, \ldots, P_n]$ of class frequencies.

**Birth rate.** The instantaneous rate of birth to organisms in the $i$-th class is $b_i P_i.$

**Death rate.** The instantaneous rate of death of organisms in the $i$-th class is $d P_i.$

**Growth rate.** The instantaneous rate of change in frequency of the $i$-th class is:

$$\frac{\text{d}P_i}{\text{d}t} = \underbrace{b_i P_i}_\text{birth rate} - \underbrace{d P_i}_\text{death rate} = m_i P_i.$$

**Fitness.** We refer to Malthusian parameter $m_i = b_i - d$ as the fitness of class $i.$ Fitnesses are spaced evenly, i.e., $m_i = i \Delta - d$ for $i = 0,$ $1, \ldots,$ $n.$ We refer to $\Delta$ as the bin width.

**Analytic solution.** Class frequencies change exponentially: $P(i, t) = P(i, 0) \exp(t m_i).$

## Animation: Evolution without mutation

In [3]:
%%time 

"""
Define times for which frequencies will be calculated.
"""
n_years = 700
times = np.linspace(0, n_years, n_years+1)

"""
Define fitness bins.
"""
death_parameter = 0.1
min_fitness = -death_parameter
max_fitness = 0.1
bin_width = 5e-4
bins = Factors.construct(min_fitness, max_fitness, bin_width)

"""
Define Gaussian-distributed initial frequencies.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0, std=0.013,
                                          crop=np.inf, density=False)

"""
Define default distribution of probability over mutational effects, with
all mass on zero effect. There is effectively no mutation.
"""
mutations = EffectsDistribution(bins)

"""
Solve numerically (integrate) for frequencies at all times.
"""
label = 'Numerical'
pop = Population(initial_frequencies, mutations, label=label, matrix=True)
frequencies, report = ivp_solution(pop, initial_frequencies, times)
print(report.message)

"""
Wrap the solution with an Evolution object to facilitate animation.
"""
ev = Evolution(pop)
ev.set_trajectory(frequencies)

"""
Evaluate analytical expression of frequencies at all times.
"""
label = 'Analytical'
pop = Population(initial_frequencies, mutations, label=label, matrix=True)
frequencies = initial_frequencies[:].T * np.exp(np.outer(times, bins.growth))
ev_analytical = Evolution(pop)
ev_analytical.set_trajectory(frequencies)

"""
Animate comparison of the numerical and analytical solutions at all times.
"""
subtitle = 'Without Mutation: Numerical vs Analytical Solution'
c = Comparison([ev, ev_analytical], subtitle, linestyles=['-', ':'])
anim = c.animate(100, effective=False, dpi=600)
filename = DIR + 'no_mutations.mp4'
save_and_display_video(anim, filename)

The solver successfully reached the interval end.


CPU times: user 1min 22s, sys: 13.4 s, total: 1min 35s
Wall time: 1min 25s


## Sketch of discrete-fitness model with mutation

Again, without mutation, the instantaneous rate of change in frequency of class $i$ is:

$$\frac{\text{d}P_i}{\text{d}t} = \underbrace{b_i P_i}_\text{birth rate} - \underbrace{d P_i.}_\text{death rate}$$

With mutation, class-$i$ organisms are born to parents in all classes $j,$ not just parents in class $i.$ We revise the expression of birth rate accordingly:

$$\frac{\text{d}P_i}{\text{d}t} = \sum_j f_{ij} b_j P_j - d P_i,$$

where $f_{ij}$ is the fraction of births to class-$j$ parents that are in class $i.$

**Note.** Organisms in class $i$ are born to parents in class $j$ at the rate $f_{ij} b_j P_j.$ The overall rate of birth of class-$i$ organisms is $\sum_j f_{ij} b_j P_j.$


## Defining the distribution of offspring over classes

Basener and Sanford botch the definition of $f_{ij}$ (Section 5) in several ways.

Their basic idea is to discretize a continuous distribution of probability over the possible effects of mutation on the fitness of the offspring, and to set 

$$f_{ij} = p(m_i - m_j) = p((i - j) \Delta),$$

where $p$ is the discrete distribution.

**Error:** $\sum_i f_{ij} < 1$ because the tails of the distribution $p$ extend out of range.

We correct this error straightforwardly by normalizing, i.e., using 

$$f^\prime_{ij} = \frac{f_{ij} }{ \sum_i f_{ij}}$$

in place of $f_{ij}.$ Then $\sum_i f^\prime_{ij} = 1$ as required.

## Evolution with Gaussian mutational effects

In [4]:
%%time 

"""
Define times for which frequencies will be calculated.
"""
n_years = 1000
times = np.linspace(0, n_years, n_years+1)

"""
Define fitness bins.
"""
death_parameter = 0.1
min_fitness = -death_parameter
max_fitness = 0.1
bin_width = 5e-4
bins = Factors.construct(min_fitness, max_fitness, bin_width)

"""
Define Gaussian-distributed initial frequencies.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0, std=0.013,
                                          crop=np.inf, density=False)

"""
Define Gaussian distribution over mutational effects as in BS Section 5.3.
"""
mutations = EffectsDistribution(bins)
mutations.gaussian(0, 0.002)
mutations.normalize()

print('Mean effect of mutation on fitness     :',
      float(mutations.mean()))
print('Deleterious-to-advantageous ratio      :',
      float(mutations.deleterious_to_advantageous()))
print('Probability of zero mutation effect    :',
      float(mutations.probability_neutral()))
print('Probability of positive mutation effect:',
      float(mutations.probability_advantageous()))

"""
Solve numerically (integrate) for frequencies at all times.
"""
label = 'Gaussian'
pop = Population(initial_frequencies, mutations, label=label, matrix=True)
frequencies, report = ivp_solution(pop, initial_frequencies, times)
print(report.message)

"""
Wrap the solution with an Evolution object to facilitate animation.
"""
ev = Evolution(pop)
ev.set_trajectory(frequencies)

"""
Animate the solution.
"""
subtitle = 'Nominally Gaussian Mutations'
c = Comparison([ev], subtitle)
anim = c.animate(100, effective=False, dpi=600)
filename = DIR + 'gaussian_mutations.mp4'
save_and_display_video(anim, filename)

Mean effect of mutation on fitness     : 0.0
Deleterious-to-advantageous ratio      : 1.0
Probability of zero mutation effect    : 0.09947644966022588
Probability of positive mutation effect: 0.4502617751698871
The solver successfully reached the interval end.


CPU times: user 2min 45s, sys: 21.3 s, total: 3min 6s
Wall time: 2min 57s


## Evolution with double-gamma mutational effects

Two runs, the first with weight 1e-3, and the second with weight 1e-9, of the positive mutational effects.

**With weight 1e-3 of positive mutational effects**

In [5]:
%%time 

"""
Define times for which frequencies will be calculated.
"""
n_years = 2000
times = np.linspace(0, n_years, n_years+1)

"""
Define fitness bins.
"""
death_parameter = 0.1
min_fitness = -death_parameter
max_fitness = 0.1
bin_width = 5e-4
bins = Factors.construct(min_fitness, max_fitness, bin_width)

"""
Define Gaussian-distributed initial frequencies.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0, std=0.013,
                                          crop=np.inf, density=False)

weight = mp_float('1e-3')
mutations = WeightedDoubleGamma(bins, weight=weight)
mutations.normalize()
    
print('Parameters of weighted double-gamma distribution')
print('    alpha :', mutations.alpha)
print('    beta  :', mutations.beta)
print('    weight:', mutations.weight)
print('Mean effect of mutation on fitness     :',
      float(mutations.mean()))
print('Deleterious-to-advantageous ratio      :',
      float(mutations.deleterious_to_advantageous()))
print('Probability of zero mutation effect    :',
      float(mutations.probability_neutral()))
print('Probability of positive mutation effect:',
      float(mutations.probability_advantageous()))

"""
Solve numerically (integrate) for frequencies at all times.
"""
label = 'Weight 1e-3'
pop = Population(initial_frequencies, mutations, label=label, matrix=True)
frequencies, report = ivp_solution(pop, initial_frequencies, times)
print(report.message)

"""
Wrap the solution with an Evolution object to facilitate animation.
"""
ev = Evolution(pop)
ev.set_trajectory(frequencies)

"""
Animate the solution.
"""
subtitle = 'Nominally Double-Gamma Mutations'
c = Comparison([ev], subtitle)
anim = c.animate(100, effective=False, dpi=600)
filename = DIR + 'gamma_1e-3_mutations.mp4'
save_and_display_video(anim, filename)

Parameters of weighted double-gamma distribution
    alpha : 0.5
    beta  : 500.0
    weight: 0.001
Mean effect of mutation on fitness     : -0.0009812564459105151
Deleterious-to-advantageous ratio      : 999.0
Probability of zero mutation effect    : 0.38292492254802624
Probability of positive mutation effect: 0.0006170750774519738
The solver successfully reached the interval end.


CPU times: user 3min 29s, sys: 37.9 s, total: 4min 7s
Wall time: 4min 2s


**With weight 1e-9 of positive mutational effects**

In [6]:
%%time 

"""
Define times for which frequencies will be calculated.
"""
n_years = 10000
times = np.linspace(0, n_years, n_years+1)

"""
Define fitness bins.
"""
death_parameter = 0.1
min_fitness = -death_parameter
max_fitness = 0.1
bin_width = 5e-4
bins = Factors.construct(min_fitness, max_fitness, bin_width)

"""
Define Gaussian-distributed initial frequencies.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0, std=0.013,
                                          crop=np.inf, density=False)

weight = mp_float('1e-9')
mutations = WeightedDoubleGamma(bins, weight=weight)
mutations.normalize()
    
print('Parameters of weighted double-gamma distribution')
print('    alpha :', mutations.alpha)
print('    beta  :', mutations.beta)
print('    weight:', mutations.weight)
print('Mean effect of mutation on fitness     :',
      float(mutations.mean()))
print('Deleterious-to-advantageous ratio      :',
      float(mutations.deleterious_to_advantageous()))
print('Probability of zero mutation effect    :',
      float(mutations.probability_neutral()))
print('Probability of positive mutation effect:',
      float(mutations.probability_advantageous()))

"""
Solve numerically (integrate) for frequencies at all times.
"""
label = 'Weight 1e-9'
pop = Population(initial_frequencies, mutations, label=label, matrix=True)
frequencies, report = ivp_solution(pop, initial_frequencies, times)
print(report.message)

"""
Wrap the solution with an Evolution object to facilitate animation.
"""
ev = Evolution(pop)
ev.set_trajectory(frequencies)

"""
Animate the solution.
"""
subtitle = 'Nominally Double-Gamma Mutations'
c = Comparison([ev], subtitle)
anim = c.animate(100, effective=False, dpi=600)
filename = DIR + 'gamma_1e-9_mutations.mp4'
save_and_display_video(anim, filename)

Parameters of weighted double-gamma distribution
    alpha : 0.5
    beta  : 500.0
    weight: 0.000000001
Mean effect of mutation on fitness     : -0.000983222889727457
Deleterious-to-advantageous ratio      : 999999999.0
Probability of zero mutation effect    : 0.38292492254802624
Probability of positive mutation effect: 6.170750774519738e-10
The solver successfully reached the interval end.


CPU times: user 16min 24s, sys: 3min 5s, total: 19min 30s
Wall time: 18min 54s


## Zeroing relatively small frequencies

Basener and Sanford do not report the results of numerical solution of their model.

In Section 5, "Numerical Simulations," they instead modify the model: if frequency $P_i$ falls below $10^{-9} \sum_j P_j,$ then it is set to zero.

They claim that this thresholding operation&nbsp;&mdash; zeroing relatively small frequencies&nbsp;&mdash; effectively holds the population size constant at $10^9.$

They implement the operation incorrectly in their software, evidently failing to recognize that a zeroed frequency cannot rise instaneously to threshold.

Implemented correctly, thresholding gives ludicrous results.

In [7]:
%%time

n_years = 5000

"""
Define fitness bins.
"""
death_parameter = 0.1
min_fitness = -death_parameter
max_fitness = 0.1
bin_width = 5e-4
bins = Factors.construct(min_fitness, max_fitness, bin_width)

"""
Define Gaussian-distributed initial frequencies as B&S do.
"""
initial_frequencies = GaussianFrequencies(bins)

"""
Define double-gamma distribution over mutational effects.
"""
weight = mp_float('1e-3')
mutations = WeightedDoubleGamma(bins, weight=weight)
mutations.normalize()

pop = Population(initial_frequencies, mutations, updates_per_year=100,
                 norm=sum, continuous=True, matrix=True, label='Thresholded')
ev = Evolution(pop, n_years)

"""
Animate the solution.
"""
subtitle = 'Double-Gamma Mutations, Thresholded Frequencies'
c = Comparison([ev], subtitle)
anim = c.animate(100, effective=False, dpi=600)
filename = DIR + 'thresholded.mp4'
save_and_display_video(anim, filename)

CPU times: user 1min 34s, sys: 14 s, total: 1min 48s
Wall time: 2min 10s
